# INESC TEC Uncertainty Framework - Colab Pro+ Report Runner
Este notebook é o ponto de entrada oficial para a execução assíncrona do projeto.

**ATENÇÃO:** O repositório irá clonar a *tag* `report-run-freeze` em vez da branch `main`.


In [ ]:
# 1. Mount Drive, Setup Environment, Clone Frozen Tag and Inject CUDA Paths
import os, sys, glob, subprocess
from google.colab import drive, userdata

drive.mount('/content/drive')
github_token = userdata.get('GITHUB_TOKEN')
!git clone --branch report-run-freeze https://{github_token}@github.com/AlexandreCGodinho/inesctec-uncertainty-framework.git
%cd inesctec-uncertainty-framework
!pip install -r requirements.txt
!pip install vllm

# Busca universal de bibliotecas CUDA com exclusão cirúrgica de conflitos de driver
caminhos_cuda = subprocess.check_output('find /usr/local /usr/lib -name "lib*.so*" 2>/dev/null | xargs -r -n 1 dirname | grep -E -v "(compat|/usr/local/cuda/lib64)" | sort -u | paste -sd ":" -', shell=True).decode().strip()
os.environ["LD_LIBRARY_PATH"] = caminhos_cuda + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("Ambiente reativado e bibliotecas NVIDIA injetadas com sucesso!")

In [ ]:
# 2. Start vLLM Server in Background with Log Redirect
import subprocess, time

comando_vllm = (
    "python3 -m vllm.entrypoints.openai.api_server "
    "--model google/gemma-4-12b-it "
    "--port 8000 "
    "--dtype bfloat16 "
    "--gpu-memory-utilization 0.90 "
    "--max-model-len 4096 > vllm.log 2>&1"
)

servidor = subprocess.Popen(comando_vllm, shell=True)
print(f"Servidor vLLM iniciado silenciosamente no background com PID: {servidor.pid}")
print("Aguardando 60 segundos para alocação inicial de pesos na GPU A100...")
time.sleep(60)

In [ ]:
# 3. Inspect vLLM Server Log
!tail -n 25 vllm.log

In [ ]:
# 4A. Ensaio Rápido (Micro-Smoke Run - N=2 / 20 min)
# Roda primeiro esta célula para validar a Emenda 2 e a factualidade:
!python3 -m src.runner --output-dir "/content/drive/MyDrive/test_micro_smoke" --sample-size 2

In [ ]:
# 5A. Auditoria e Resumo Estatístico do Micro-Smoke
import json, glob, pandas as pd
pasta_teste = "/content/drive/MyDrive/test_micro_smoke"

print("=== AUDITORIA DO MANIFESTO (MICRO-SMOKE) ===")
with open(f"{pasta_teste}/manifest.json", "r", encoding="utf-8") as f:
    manifesto = json.load(f)

for run_id, info in manifesto.get("runs", {}).items():
    status = "OK" if info.get("actual") == info.get("expected") else "ERRO"
    print(f"[{status}] {run_id}: {info.get('actual')}/{info.get('expected')} queries")

print("
=== DESEMPENHO E INCERTEZA POR PADRÃO (MICRO-SMOKE) ===")
arquivos = glob.glob(f"{pasta_teste}/*_pattern*.jsonl")
dados = []
for arq in arquivos:
    with open(arq, "r", encoding="utf-8") as f:
        for linha in f:
            if linha.strip():
                obj = json.loads(linha)
                padrão = obj.get("pattern_id", arq.split("_")[-1].replace(".jsonl", ""))
                latencia = obj.get("latency_seconds", 0)
                entropia = obj.get("uncertainty_metrics", {}).get("semantic_entropy", 0)
                dados.append({"Padrão": padrão, "Latência Média (s)": latencia, "Entropia Semântica": entropia})
if dados:
    df = pd.DataFrame(dados)
    resumo = df.groupby("Padrão")[["Latência Média (s)", "Entropia Semântica"]].mean().reset_index()
    print(resumo.to_string(index=False))

In [ ]:
# 6B. Rodada Oficial de Produção (Report Run - N=150 / 25 Horas)
# ATENÇÃO: Só execute esta célula após aprovar os resultados do Micro-Smoke acima!
!python3 -m src.runner --output-dir "/content/drive/MyDrive/report_run_a100"

In [ ]:
# 7B. Auditoria Final da Rodada Oficial e Transferência ZIP para o Mac
import json, glob, shutil, pandas as pd
from google.colab import files
pasta_drive = "/content/drive/MyDrive/report_run_a100"

print("=== AUDITORIA DO MANIFESTO GLOBAL (PRODUÇÃO) ===")
with open(f"{pasta_drive}/manifest.json", "r", encoding="utf-8") as f:
    manifesto = json.load(f)

for run_id, info in manifesto.get("runs", {}).items():
    status = "OK" if info.get("actual") == info.get("expected") else "ERRO"
    print(f"[{status}] {run_id}: {info.get('actual')}/{info.get('expected')} queries")

print("
=== DESEMPENHO E INCERTEZA POR PADRÃO (PRODUÇÃO) ===")
arquivos = glob.glob(f"{pasta_drive}/*_pattern*.jsonl")
dados = []
for arq in arquivos:
    with open(arq, "r", encoding="utf-8") as f:
        for linha in f:
            if linha.strip():
                obj = json.loads(linha)
                padrão = obj.get("pattern_id", arq.split("_")[-1].replace(".jsonl", ""))
                latencia = obj.get("latency_seconds", 0)
                entropia = obj.get("uncertainty_metrics", {}).get("semantic_entropy", 0)
                dados.append({"Padrão": padrão, "Latência Média (s)": latencia, "Entropia Semântica": entropia})
if dados:
    df = pd.DataFrame(dados)
    resumo = df.groupby("Padrão")[["Latência Média (s)", "Entropia Semântica"]].mean().reset_index()
    print(resumo.to_string(index=False))

print("
=== A GERAR ARQUIVO ZIP PARA O MAC ===")
shutil.make_archive("report_run_a100_telemetry", "zip", pasta_drive)
files.download("report_run_a100_telemetry.zip")
print("Download iniciado com êxito!")